In [1]:
import brainsss
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
%matplotlib inline
#from sklearn.cluster import AgglomerativeClustering
import scipy
import time
import h5py
import ants
import nibabel as nib
from scipy.ndimage import uniform_filter, gaussian_filter
import shutil
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph
import gc
import sys
import warnings
from scipy.ndimage import gaussian_filter1d,gaussian_filter
from scipy.signal import butter, sosfiltfilt, filtfilt, freqz,iirnotch
import cv2
from scipy.ndimage.morphology import binary_erosion
from scipy.ndimage.morphology import binary_dilation
from scipy.ndimage import zoom
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import sklearn
import pickle
import itertools
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import pandas as pd
from skimage import measure
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.ndimage.morphology import binary_erosion, binary_dilation
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist
import csv
from functools import reduce
from sklearn.decomposition import PCA
import multiprocessing as mp
from multiprocessing import Pool
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.multiclass import OneVsRestClassifier
import scour.scour
import gzip

In [2]:
color_map_color_1=np.asarray(['#780000','#C1121F','#E0898F','#F0C4C7','#FFFFFF','#D9E6EF','#B3CDDE','#669BBC', '#003049'])[::-1]
cmap_personal = matplotlib.colors.LinearSegmentedColormap.from_list(
    'cmap', color_map_color_1)

In [3]:
fly_num=250
ch_num=2
later_path = '/oak/stanford/groups/trc/data/Ilana/2P/data/later/'
dff_path = f'/oak/stanford/groups/trc/data/Ilana/2P/data/fly_{fly_num}/dff'
warp_path = f'/oak/stanford/groups/trc/data/Ilana/2P/data/fly_{fly_num}/warp'
cluster_dir= os.path.join(later_path,'temp_filter', 'clustering')

event1='fixed_10flies_final'
event2='fixed_best_final'

In [4]:
total_path = os.path.join(later_path, f'{event1}_event_times_split_dic.pkl')
with open(total_path, 'rb') as file:
    total_data_dict1 = pickle.load(file)
total_path = os.path.join(later_path, f'{event2}_event_times_split_dic.pkl')
with open(total_path, 'rb') as file:
    total_data_dict2 = pickle.load(file)

In [5]:
np.shape(total_data_dict1['250']['total'])

(199,)

In [6]:
np.shape(total_data_dict2['250']['total'])

(199,)

In [7]:
save_file=os.path.join(later_path,f'{fly_num}_individual_clusters_new_ch_{ch_num}_dict.pkl')
for file in os.listdir(dff_path):
    if f'_{ch_num}_' in file:
        file_path=os.path.join(dff_path,file)
for file in os.listdir(warp_path):
    if 'timestamps' in file:
        ts_path=os.path.join(warp_path,file)


In [8]:
with open(save_file, 'rb') as file:
    new_ind = pickle.load(file)

In [9]:
new_ind[0].keys()

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196])

In [10]:
n_clusters=10
event_time_bins=[]
for event in total_data_dict1[str(fly_num)]['total'][:198]:
    seconds_before = 2
    ms_per_unit = 10
    units_to_subtract = (seconds_before * 1000) // ms_per_unit
    new_timepoint = event - units_to_subtract
    event_edges=[new_timepoint,event]
    event_time_bins.append(event_edges)
event_time_bins=np.asarray(event_time_bins)

time_bins = event_time_bins
cluster_brains = {}
range_r=np.arange(n_clusters)

In [11]:
np.shape(event_time_bins)

(198, 2)

In [12]:
clus=500

for file in os.listdir(cluster_dir):
#     print(file)
    if f'_{clus}' in file and 'labels' in file and 'total' in file:
        label_files=os.path.join(cluster_dir,file)
giant_total_labels=np.load(label_files)

In [13]:
giant_total_labels=giant_total_labels.reshape(314,146,91)

In [14]:
events=total_data_dict1[str(fly_num)]['total']

In [15]:
%%time
with h5py.File(file_path, 'r') as hf, h5py.File(ts_path, 'r') as tf:
    data_ds = hf['data'][:]
    time_ds = tf['data'][:]

   

CPU times: user 0 ns, sys: 1min 1s, total: 1min 1s
Wall time: 7min 18s


In [16]:
cluster=250
mask = (giant_total_labels == cluster)
x_idx, y_idx, z_idx = np.where(mask)
n_voxels = len(x_idx)
print(n_voxels)

1675


In [17]:
voxel_data_cache = []
voxel_times_cache = []

for x, y, z in zip(x_idx, y_idx, z_idx):
    voxel_data_cache.append(data_ds[x, y, z, :])
    voxel_times_cache.append(time_ds[x, y, z, :])



In [18]:
# Convert to numpy arrays for faster processing
voxel_data_cache = np.array(voxel_data_cache)  # Shape: (n_voxels, timepoints)
voxel_times_cache = np.array(voxel_times_cache) 
cluster_brains[cluster] = {}
print(voxel_data_cache.shape)
print(voxel_times_cache.shape)

(1675, 3384)
(1675, 3384)
